# ASL Citizen — Data Analysis

Explores the **top-100 class subset of ASL Citizen** used to train the isolated sign recognition model (Phase-aware TCN).

**Two parts:**
- **Part A** (runs anywhere, no download): analysis from the label map CSV already in the repo.
- **Part B** (requires the dataset zip on Drive): loads the official ASL Citizen split CSVs, verifies signer independence, and cross-checks counts against our label map.

Run **Part A only** if you just want the split/class-distribution analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

---
## Part A — Label Map Analysis
Uses `data/recognition_label_map_with_split_counts.csv` which is committed to the repo.

Columns:
| Column | Meaning |
|---|---|
| `rec_label_id` | 0-99 class index used by the model |
| `gloss` | ASL sign name |
| `train_count` / `val_count` / `test_count` | raw counts from official ASL Citizen splits |
| `rec_train_count` / `rec_val_count` / `rec_test_count` | post-phase-filter counts (what the model actually sees) |

In [ ]:
# Works whether you run from repo root or from notebooks/
HERE = Path('__file__').resolve().parent if '__file__' in dir() else Path.cwd()
REPO_ROOT = HERE.parent if HERE.name == 'notebooks' else HERE
LABEL_MAP_PATH = REPO_ROOT / 'data' / 'recognition_label_map_with_split_counts.csv'

df = pd.read_csv(LABEL_MAP_PATH)
print(f"Classes: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
df.head(10)

### A1 — Split summary

In [ ]:
raw  = {'Train': df['train_count'].sum(),  'Val': df['val_count'].sum(),  'Test': df['test_count'].sum()}
filt = {'Train': df['rec_train_count'].sum(), 'Val': df['rec_val_count'].sum(), 'Test': df['rec_test_count'].sum()}

summary = pd.DataFrame({'Raw (pre-filter)': raw, 'Post-filter (model sees)': filt})
summary.loc['Total'] = summary.sum()
summary['Dropped'] = summary['Raw (pre-filter)'] - summary['Post-filter (model sees)']
print(summary.to_string())
print(f"\nPhase-filter drop rate: {summary.loc['Total','Dropped'] / summary.loc['Total','Raw (pre-filter)'] * 100:.1f}%")

In [ ]:
# Split proportion pie
sizes = [filt['Train'], filt['Val'], filt['Test']]
labels = [f"Train\n{filt['Train']} ({filt['Train']/sum(sizes)*100:.1f}%)",
          f"Val\n{filt['Val']} ({filt['Val']/sum(sizes)*100:.1f}%)",
          f"Test\n{filt['Test']} ({filt['Test']/sum(sizes)*100:.1f}%)"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].pie(sizes, labels=labels, colors=['#4c72b0','#dd8452','#55a868'],
            startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[0].set_title('Post-filter split distribution\n(100 classes, 2,234 samples total)')

# Raw vs filtered bar
x = np.arange(3)
w = 0.35
axes[1].bar(x - w/2, list(raw.values()),  w, label='Raw', color='#4c72b0', alpha=0.8)
axes[1].bar(x + w/2, list(filt.values()), w, label='Post-filter', color='#55a868', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Train', 'Val', 'Test'])
axes[1].set_ylabel('Sample count')
axes[1].set_title('Raw vs post-phase-filter counts')
axes[1].legend()
axes[1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f'{int(v):,}'))

plt.tight_layout()
plt.show()

### A2 — Class imbalance

In [ ]:
df_sorted = df.sort_values('rec_total_count', ascending=False).reset_index(drop=True)

print("=== Per-class sample statistics (post-filter) ===")
for split, col in [('Train','rec_train_count'), ('Val','rec_val_count'), ('Test','rec_test_count'), ('Total','rec_total_count')]:
    s = df[col]
    print(f"  {split:<6}  min={s.min():>3}  median={s.median():>5.1f}  mean={s.mean():>5.1f}  max={s.max():>3}  std={s.std():>4.1f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

colors_train = ['#4c72b0'] * len(df_sorted)
colors_test  = ['#dd8452'] * len(df_sorted)

axes[0].bar(df_sorted['gloss'], df_sorted['rec_train_count'], color='#4c72b0', alpha=0.85, label='Train')
axes[0].bar(df_sorted['gloss'], df_sorted['rec_val_count'],   color='#55a868', alpha=0.85, label='Val',  bottom=df_sorted['rec_train_count'])
axes[0].bar(df_sorted['gloss'], df_sorted['rec_test_count'],  color='#dd8452', alpha=0.85, label='Test', bottom=df_sorted['rec_train_count'] + df_sorted['rec_val_count'])
axes[0].set_ylabel('Samples')
axes[0].set_title('Sample count per class (sorted by total, post-phase-filter)')
axes[0].legend()
axes[0].axhline(df['rec_total_count'].mean(), color='black', ls='--', lw=1, label=f'Mean={df["rec_total_count"].mean():.1f}')

# Highlight min/max
ax = axes[0]
ax.annotate(df_sorted.iloc[0]['gloss'], xy=(0, df_sorted.iloc[0]['rec_total_count']),
            xytext=(3, df_sorted.iloc[0]['rec_total_count'] + 1), fontsize=8, color='navy')
ax.annotate(df_sorted.iloc[-1]['gloss'], xy=(len(df_sorted)-1, df_sorted.iloc[-1]['rec_total_count']),
            xytext=(len(df_sorted)-12, df_sorted.iloc[-1]['rec_total_count'] + 2), fontsize=8, color='darkred')

# Train/test ratio per class
ratio = df_sorted['rec_train_count'] / df_sorted['rec_test_count'].replace(0, np.nan)
axes[1].bar(df_sorted['gloss'], ratio, color='#9467bd', alpha=0.75)
axes[1].axhline(ratio.mean(), color='black', ls='--', lw=1)
axes[1].set_ylabel('Train / Test ratio')
axes[1].set_title('Train-to-test ratio per class (balanced = ~1.4 expected)')

for ax in axes:
    ax.tick_params(axis='x', rotation=90, labelsize=6.5)

plt.tight_layout()
plt.show()

In [ ]:
# Histogram of total counts
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['rec_total_count'], bins=15, color='#4c72b0', edgecolor='white', alpha=0.85)
ax.axvline(df['rec_total_count'].mean(),   color='red',    ls='--', label=f'Mean  {df["rec_total_count"].mean():.1f}')
ax.axvline(df['rec_total_count'].median(), color='orange', ls='--', label=f'Median {df["rec_total_count"].median():.0f}')
ax.set_xlabel('Total samples per class (post-filter)')
ax.set_ylabel('Number of classes')
ax.set_title('Class-size distribution across all 100 glosses')
ax.legend()
plt.tight_layout()
plt.show()

### A3 — Phase-filter impact per class

Classes where `rec_train_count < train_count` lost samples during phase-quality filtering.
These are typically signs with multi-peak or oscillatory motion (e.g., TWINS, COMB, PIPE)
where the velocity-heuristic pseudo-labeller fails.

In [ ]:
df['train_dropped'] = df['train_count'] - df['rec_train_count']
df['val_dropped']   = df['val_count']   - df['rec_val_count']
df['test_dropped']  = df['test_count']  - df['rec_test_count']
df['total_dropped'] = df['total_count'] - df['rec_total_count']

affected = df[df['total_dropped'] > 0].sort_values('total_dropped', ascending=False)
print(f"Classes with at least one filtered sample: {len(affected)} / {len(df)}")
print()
print(affected[['gloss','train_dropped','val_dropped','test_dropped','total_dropped']].to_string(index=False))

In [ ]:
if len(affected) > 0:
    fig, ax = plt.subplots(figsize=(max(6, len(affected)*0.6), 4))
    x = np.arange(len(affected))
    ax.bar(x, affected['train_dropped'], label='Train', color='#4c72b0', alpha=0.85)
    ax.bar(x, affected['val_dropped'],   label='Val',   color='#55a868', alpha=0.85, bottom=affected['train_dropped'])
    ax.bar(x, affected['test_dropped'],  label='Test',  color='#dd8452', alpha=0.85,
           bottom=affected['train_dropped'].values + affected['val_dropped'].values)
    ax.set_xticks(x)
    ax.set_xticklabels(affected['gloss'].tolist(), rotation=45, ha='right')
    ax.set_ylabel('Samples dropped')
    ax.set_title('Samples removed by phase-quality filter per class')
    ax.legend()
    plt.tight_layout()
    plt.show()

### A4 — Label ID assignment
Verify the `rec_label_id` assignment is sequential and matches sorted alphabetical order.

In [ ]:
assert list(df['rec_label_id']) == list(range(len(df))), "rec_label_id is not sequential 0..99"
print(f"rec_label_id: sequential 0 to {len(df)-1} — OK")

alpha_order = sorted(df['gloss'].tolist())
is_alpha = df['gloss'].tolist() == alpha_order
print(f"Glosses are in alphabetical order: {is_alpha}")

print(f"\nFirst 5 classes:")
for _, row in df.head(5).iterrows():
    print(f"  [{int(row['rec_label_id']):>2}]  {row['gloss']:<30}  train={int(row['rec_train_count'])}  val={int(row['rec_val_count'])}  test={int(row['rec_test_count'])}")
print("  ...")
print(f"Last 5 classes:")
for _, row in df.tail(5).iterrows():
    print(f"  [{int(row['rec_label_id']):>2}]  {row['gloss']:<30}  train={int(row['rec_train_count'])}  val={int(row['rec_val_count'])}  test={int(row['rec_test_count'])}")

---
## Part B — Full Dataset Exploration

**Requires the ASL Citizen zip on Google Drive.**

Skip to the bottom summary if you only want the label map analysis above.

Download: https://www.microsoft.com/en-us/research/project/asl-citizen/  
File: `ASL_Citizen.zip` (~20 GB)

In [ ]:
# ── Colab: mount Drive ──────────────────────────────────────────────────────
# Comment out if running locally with the zip already extracted.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile, os

ZIP_PATH  = Path('/content/drive/MyDrive/CAPSTONE_ASL/download/ASL_Citizen.zip')
DATA_DIR  = Path('/content/drive/MyDrive/CAPSTONE_ASL/ASL_Citizen')

# ── Or set a local path if you've already extracted it ──────────────────────
# ZIP_PATH = Path('/path/to/ASL_Citizen.zip')
# DATA_DIR = Path('/path/to/ASL_Citizen')

print(f"Zip exists:  {ZIP_PATH.exists()}")
if ZIP_PATH.exists():
    print(f"Zip size:    {ZIP_PATH.stat().st_size / 1e9:.1f} GB")

In [ ]:
# Extract just the three split CSVs (fast — a few KB each)
SPLITS_DIR = DATA_DIR / 'splits'

if not (SPLITS_DIR / 'train.csv').exists():
    print("Extracting split CSVs...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        for name in ['ASL_Citizen/splits/train.csv',
                     'ASL_Citizen/splits/val.csv',
                     'ASL_Citizen/splits/test.csv']:
            z.extract(name, path=DATA_DIR.parent)
    print("Done.")
else:
    print("Split CSVs already extracted.")

print("\nFiles in splits/:")
for f in sorted(SPLITS_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

### B1 — Load official splits

In [ ]:
GLOSS_COL  = 'Gloss'
SIGNER_COL = 'Participant ID'
VIDEO_COL  = 'Video file'

train_df = pd.read_csv(SPLITS_DIR / 'train.csv')
val_df   = pd.read_csv(SPLITS_DIR / 'val.csv')
test_df  = pd.read_csv(SPLITS_DIR / 'test.csv')

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"{name}: {len(split):>6} rows  |  {split[GLOSS_COL].nunique():>4} classes  |  {split[SIGNER_COL].nunique():>2} signers")

print(f"\nColumns: {train_df.columns.tolist()}")
print()
train_df.head(5)

### B2 — Signer independence

In [ ]:
tr_sig = set(train_df[SIGNER_COL].unique())
va_sig = set(val_df[SIGNER_COL].unique())
te_sig = set(test_df[SIGNER_COL].unique())

print("Signer counts per split:")
print(f"  Train: {len(tr_sig)}  Val: {len(va_sig)}  Test: {len(te_sig)}  Union: {len(tr_sig | va_sig | te_sig)}")
print()
print("Signer overlap (signer-independent = 0 overlap between test and others):")
print(f"  Train ∩ Val:   {len(tr_sig & va_sig)} signers")
print(f"  Train ∩ Test:  {len(tr_sig & te_sig)} signers")
print(f"  Val   ∩ Test:  {len(va_sig & te_sig)} signers")

# Visualise
fig, ax = plt.subplots(figsize=(7, 4))
all_signers = sorted(tr_sig | va_sig | te_sig)
mat = np.array([
    [int(s in tr_sig) for s in all_signers],
    [int(s in va_sig) for s in all_signers],
    [int(s in te_sig) for s in all_signers],
])
ax.imshow(mat, aspect='auto', cmap='Blues', interpolation='nearest')
ax.set_yticks([0,1,2])
ax.set_yticklabels(['Train','Val','Test'])
ax.set_xlabel('Signer index (sorted)')
ax.set_title('Signer presence across splits (blue = present)')
plt.tight_layout()
plt.show()

### B3 — Top-100 subset selection

In [ ]:
# Top-100 classes by training frequency
class_counts_full = train_df[GLOSS_COL].value_counts()
print(f"Full vocabulary: {len(class_counts_full)} classes in train")

top100_glosses = set(class_counts_full.head(100).index)

sub_train = train_df[train_df[GLOSS_COL].isin(top100_glosses)].reset_index(drop=True)
sub_val   = val_df[val_df[GLOSS_COL].isin(top100_glosses)].reset_index(drop=True)
sub_test  = test_df[test_df[GLOSS_COL].isin(top100_glosses)].reset_index(drop=True)

print(f"\nTop-100 subset:")
print(f"  Train: {len(sub_train)} videos  ({sub_train[GLOSS_COL].nunique()} classes)")
print(f"  Val:   {len(sub_val)} videos  ({sub_val[GLOSS_COL].nunique()} classes)")
print(f"  Test:  {len(sub_test)} videos  ({sub_test[GLOSS_COL].nunique()} classes)")
print(f"  Total: {len(sub_train)+len(sub_val)+len(sub_test)} videos")

# Classes present in all three splits
all3 = set(sub_train[GLOSS_COL]) & set(sub_val[GLOSS_COL]) & set(sub_test[GLOSS_COL])
print(f"\nClasses present in all three splits: {len(all3)} / 100")

### B4 — Cross-check against our label map

In [ ]:
# Build counts from official CSVs and compare with our CSV
raw_counts = pd.DataFrame({
    'gloss':       sorted(top100_glosses),
    'tr_official': [sub_train[GLOSS_COL].value_counts().get(g, 0) for g in sorted(top100_glosses)],
    'va_official': [sub_val[GLOSS_COL].value_counts().get(g, 0)   for g in sorted(top100_glosses)],
    'te_official': [sub_test[GLOSS_COL].value_counts().get(g, 0)  for g in sorted(top100_glosses)],
})

merged = df.merge(raw_counts, on='gloss')
mismatches_tr = (merged['train_count'] != merged['tr_official']).sum()
mismatches_va = (merged['val_count']   != merged['va_official']).sum()
mismatches_te = (merged['test_count']  != merged['te_official']).sum()

print("Cross-check: label map CSV  vs  official split CSVs")
print(f"  Train mismatches: {mismatches_tr}")
print(f"  Val   mismatches: {mismatches_va}")
print(f"  Test  mismatches: {mismatches_te}")
print()

if mismatches_tr + mismatches_va + mismatches_te == 0:
    print("All counts match — label map is consistent with the official splits.")
else:
    print("Mismatches found:")
    mask = (merged['train_count'] != merged['tr_official']) | \
           (merged['val_count']   != merged['va_official']) | \
           (merged['test_count']  != merged['te_official'])
    print(merged[mask][['gloss','train_count','tr_official','val_count','va_official','test_count','te_official']])

### B5 — Video file inventory

In [ ]:
# How many unique video files does our subset need?
all_subset = pd.concat([sub_train, sub_val, sub_test], ignore_index=True)
unique_videos = all_subset[VIDEO_COL].unique()
print(f"Unique video files needed for top-100 subset: {len(unique_videos):,}")
print(f"Example filenames:")
for v in unique_videos[:5]:
    print(f"  {v}")

In [ ]:
# Check how many of those are already extracted on Drive
VIDEO_DIR = DATA_DIR / 'videos'   # adjust if your extraction path differs

if VIDEO_DIR.exists():
    extracted = {p.name for p in VIDEO_DIR.rglob('*.mp4')}
    needed    = set(Path(v).name for v in unique_videos)
    found     = needed & extracted
    missing   = needed - extracted
    print(f"Extracted:  {len(extracted):,} mp4 files in {VIDEO_DIR}")
    print(f"Needed:     {len(needed):,}")
    print(f"Found:      {len(found):,}")
    print(f"Missing:    {len(missing):,}")
else:
    print(f"Video directory not found at {VIDEO_DIR}")
    print("Run the extraction cell from Feature.ipynb first.")

### B6 — Samples per signer in our subset

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, (name, split) in zip(axes, [('Train', sub_train), ('Val', sub_val), ('Test', sub_test)]):
    signer_counts = split[SIGNER_COL].value_counts().sort_values(ascending=False)
    ax.bar(range(len(signer_counts)), signer_counts.values, color='#4c72b0', alpha=0.8)
    ax.set_xlabel('Signer rank')
    ax.set_ylabel('Videos')
    ax.set_title(f'{name} — {len(signer_counts)} signers')
    ax.axhline(signer_counts.mean(), color='red', ls='--', lw=1,
               label=f'mean {signer_counts.mean():.1f}')
    ax.legend(fontsize=8)

plt.suptitle('Videos per signer across splits (top-100 subset)', y=1.01)
plt.tight_layout()
plt.show()

---
## Summary

| Metric | Value |
|---|---|
| Classes | 100 |
| Train samples (post-filter) | 1,215 |
| Val samples (post-filter) | 242 |
| Test samples (post-filter) | 777 |
| Total (post-filter) | 2,234 |
| Samples filtered by phase quality | see A3 |
| Signer-independent splits | Yes |

**What the model pipeline expects (per sample):**
- A short MP4 clip of one signer performing one sign.
- MediaPipe extracts 49 landmarks (7 pose + 21 × 2 hands) per frame.
- 30 frames sampled uniformly → 444-d motion feature vector per frame.
- Phase TCN assigns a phase label (background / preparation / stroke / retraction).
- Active (stroke) region cropped and passed to Recognition TCN with 4 phase one-hots appended → 448-d.
- Softmax over 100 classes.